<a href="https://colab.research.google.com/github/lonerkou/ai/blob/main/4.keras_CNN_mnist.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ==========================================
# 1. 匯入必要的模組與函式庫
# ==========================================
from tensorflow import keras
from keras.models import Sequential
from keras.layers import Dense, Conv2D, MaxPooling2D, Flatten
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# ==========================================
# 2. 載入 MNIST 手寫數字資料集與前處理
# ==========================================
mnist = keras.datasets.mnist
(train_data, train_label), (test_data, test_label) = mnist.load_data()

# 保留原始測試集標籤（1D 數字陣列），供後續製作混淆矩陣使用
test_label_original = test_label

# 【關鍵步驟 1：正規化】將像素值從 0~255 縮放到 0.0~1.0 之間，加速模型收斂並提升準確率
train_data = train_data / 255.0
test_data = test_data / 255.0

# 【關鍵步驟 2：調整形狀】CNN 卷積層需要輸入四維張量 (樣本數, 高, 寬, 通道數)
# 由於 MNIST 是單通道黑白圖，因此將形狀從 (60000, 28, 28) 轉為 (60000, 28, 28, 1)
train_data = train_data.reshape(-1, 28, 28, 1)
test_data = test_data.reshape(-1, 28, 28, 1)

# 將標籤轉換為 One-hot Encoding 矩陣 (例如 5 -> [0, 0, 0, 0, 0, 1, 0, 0, 0, 0])
train_label = keras.utils.to_categorical(train_label)
test_label = keras.utils.to_categorical(test_label)

print("訓練資料處理後形狀:", train_data.shape)
print("訓練標籤處理後形狀:", train_label.shape)
print("-" * 40)


# ==========================================
# 3. 建立 CNN 卷積神經網路模型架構
# ==========================================
model = Sequential()

# 第一層：卷積層 (Conv2D) - 萃取影像特徵
# filters=16: 16個濾鏡, kernel_size=(5,5): 5x5卷積核, input_shape: 28x28x1
model.add(Conv2D(filters=16, kernel_size=(5, 5), input_shape=(28, 28, 1), activation='relu'))

# 第二層：最大池化層 (MaxPooling2D) - 縮減特徵圖尺寸，降低運算量
model.add(MaxPooling2D(pool_size=(2, 2)))

# 第三層：平坦層 (Flatten) - 將二維特徵圖轉為一維向量
model.add(Flatten())

# 第四層：輸出層 (Dense) - 10個神經元對應數字 0~9，使用 softmax 計算機率分佈
model.add(Dense(10, activation='softmax'))

# 印出模型結構摘要
model.summary()

# 編譯模型
model.compile(optimizer='rmsprop',
              loss='categorical_crossentropy',
              metrics=['accuracy'])


# ==========================================
# 4. 訓練模型 (加入驗證集監控過擬合)
# ==========================================
# validation_split=0.2: 自動切出 20% 的訓練資料作為驗證集
train_history = model.fit(
    train_data,
    train_label,
    epochs=10,
    batch_size=512,
    validation_split=0.2
)


# ==========================================
# 5. 評估模型效能
# ==========================================
score = model.evaluate(test_data, test_label)
print(f"\n測試集上的準確率 (Accuracy): {score[1]*100:.2f}%")


# ==========================================
# 6. 視覺化訓練歷史與混淆矩陣分析
# ==========================================

# 定義繪製訓練歷程的函式
def show_train_history(history, train_key, val_key):
    plt.plot(history.history[train_key])
    plt.plot(history.history[val_key])
    plt.title('Train History')
    plt.ylabel(train_key.capitalize())
    plt.xlabel('Epoch')
    plt.legend(['train', 'validation'], loc='upper left')
    plt.show()

# 繪製準確率與損失值變化圖
print("\n--- 繪製訓練歷史圖表 ---")
show_train_history(train_history, 'accuracy', 'val_accuracy')
show_train_history(train_history, 'loss', 'val_loss')

# 計算混淆矩陣 (Confusion Matrix)
print("\n--- 混淆矩陣交叉比對表 ---")
prediction = model.predict(test_data)
prediction_label = np.argmax(prediction, axis=1)

# 使用 Pandas 建立真實標籤與預測結果的交叉對照表
crosstab_result = pd.crosstab(
    test_label_original,
    prediction_label,
    rownames=['真實標籤 (Label)'],
    colnames=['預測結果 (Predict)']
)
display(crosstab_result)

訓練資料處理後形狀: (60000, 28, 28, 1)
訓練標籤處理後形狀: (60000, 10)
----------------------------------------


/usr/local/lib/python3.13/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 24, 24, 16)     │           416 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 12, 12, 16)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_1 (Flatten)             │ (None, 2304)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 10)             │        23,050 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 23,466 (91.66 KB)

 Trainable params: 23,466 (91.66 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/10
94/94 ━━━━━━━━━━━━━━━━━━━━ 17s 170ms/step - accuracy: 0.8300 - loss: 0.6938 - val_accuracy: 0.9183 - val_loss: 0.2951
Epoch 2/10
94/94 ━━━━━━━━━━━━━━━━━━━━ 16s 168ms/step - accuracy: 0.9201 - loss: 0.2733 - val_accuracy: 0.9402 - val_loss: 0.2227
Epoch 3/10
